In [ ]:
# ============================================================
# WEEK 7| June 29 (Monday)
# Task: Automated directory testing, pipeline stability check,
#       and final GitHub push preparation
# Environment: Google Colab
# ============================================================

# ----- STEP 0: Mount Google Drive -----
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import json
from datetime import datetime

# ----- STEP 1: Define Project Root -----
# 🔧 Update this path to match your Drive folder
PROJECT_ROOT = "/content/drive/MyDrive/HCL_Project"

# Expected directory structure from Week 1
EXPECTED_DIRS = [
    "data/raw",
    "data/processed",
    "data/metadata",
    "notebooks",
    "visualizations"
]


# ----- STEP 2: Automated Directory Testing -----
def test_directory_structure(root, expected):
    print("=" * 55)
    print(" DIRECTORY STRUCTURE TEST")
    print("=" * 55)
    results = {}
    for d in expected:
        full_path = os.path.join(root, d)
        exists = os.path.isdir(full_path)
        results[d] = exists
        status = "✅ FOUND" if exists else "❌ MISSING"
        print(f"{status:12} -> {d}")
    return results


# ----- STEP 3: Verify Key Output Files Exist -----
def test_key_files(root):
    print("\n" + "=" * 55)
    print(" KEY OUTPUT FILE TEST")
    print("=" * 55)
    key_files = [
        "data/processed/master_table.parquet",
        "data/processed/final_predictions.csv",
        "data/metadata/cleaning_thresholds.json"
    ]
    results = {}
    for f in key_files:
        full_path = os.path.join(root, f)
        exists = os.path.isfile(full_path)
        results[f] = exists
        status = "✅ FOUND" if exists else "⚠️ NOT FOUND"
        print(f"{status:14} -> {f}")
    return results


# ----- STEP 4: Pipeline Stability Check -----
def test_pipeline_stability(root):
    """Loads the master table and runs basic integrity assertions."""
    print("\n" + "=" * 55)
    print(" PIPELINE STABILITY CHECK")
    print("=" * 55)
    master_path = os.path.join(root, "data/processed/master_table.parquet")

    if not os.path.isfile(master_path):
        print("⚠️ Master table not found. Skipping stability check.")
        return False
    try:
        df = pd.read_csv(master_path) if master_path.endswith(".csv") \
            else pd.read_parquet(master_path)
        print(f"✅ Master table loaded: {df.shape[0]} rows, {df.shape[1]} cols")

        # Integrity assertions
        assert df.shape[0] > 0, "Master table is empty!"
        null_pct = (df.isnull().sum().sum() / df.size) * 100
        print(f"   - Total null percentage: {null_pct:.2f}%")
        print(f"   - Columns: {list(df.columns)}")
        print("✅ Pipeline stability test PASSED")
        return True
    except Exception as e:
        print(f"❌ Pipeline stability test FAILED: {e}")
        return False


# ----- STEP 5: Generate Hand-off Test Report -----
def generate_test_report(dir_res, file_res, pipe_ok, root):
    report = {
        "test_date": str(datetime.now()),
        "task": "June 29 - Codebase Testing & Hand-off Prep",
        "directory_check": dir_res,
        "file_check": file_res,
        "pipeline_stable": pipe_ok
    }
    out_path = os.path.join(root, "data/metadata/june29_test_report.json")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(report, f, indent=4)
    print(f"\n📁 Test report saved to: {out_path}")


# ----- STEP 6: GitHub Push Helper (run in a Colab cell) -----
def github_push_instructions():
    print("\n" + "=" * 55)
    print(" GITHUB PUSH (run these in a Colab cell with '!')")
    print("=" * 55)
    print("""
!cd /content/drive/MyDrive/HCL_Project && \\
 git add . && \\
 git commit -m "Final codebase - June 29 testing complete" && \\
 git push origin main
""")


# ============================================================
# RUN ALL TESTS
# ============================================================
if __name__ == "__main__":
    dir_results = test_directory_structure(PROJECT_ROOT, EXPECTED_DIRS)
    file_results = test_key_files(PROJECT_ROOT)
    pipeline_ok = test_pipeline_stability(PROJECT_ROOT)
    generate_test_report(dir_results, file_results, pipeline_ok, PROJECT_ROOT)
    github_push_instructions()
    print("\n🎉 June 29 tasks executed.")